# 6. Reproduce full-frame UMA forces
**Input:** original periodic dump, local uma-s-1p1 model, units, task, full-frame charge/spin and tolerance.
**Output:** errors, provenance, model hash, timings and optional JSON.
**Module:** force_validation → lammpsio and local UMA calculator.

This checks numerical reproduction, not MLIP accuracy or barrier validity. Use the full periodic frame, not an extracted pair. Weights are not bundled or downloaded. Inference is off until RUN_VALIDATION=True.

Set `RUN_DIR` to your simulation folder. Leave `INPUT_FILE`, `DUMP_FILE`, and `DATA_FILE` as `None` for automatic discovery. Files directly in the folder take priority; subfolders are searched if necessary. If several files match, set the corresponding filename override. Keep units and other physical settings explicit.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
from MartiniFuncs.trajectory_analysis import resolve_run_files
from MartiniFuncs.force_validation import validate_forces, MD_INFERENCE_SETTINGS

RUN_DIR = Path(r"/path/to/simulation")
INPUT_FILE = None  # optional filename override; None discovers automatically
DUMP_FILE = None
DATA_FILE = None
UNITS = "metal"
CHECKPOINT = Path(r"/path/to/uma-s-1p1.pt")
TASK = "omol"
CHARGE = None  # full-frame total charge
SPIN_MULTIPLICITY = None  # multiplicity, not spin quantum number
FRAME_INDICES = [0]
ATOL_EV_PER_ANGSTROM = 1e-4  # illustrative; choose your reproduction tolerance
DEVICE = "cuda"
RUN_VALIDATION = False
REPORT_FILE = None
INFERENCE_SETTINGS = MD_INFERENCE_SETTINGS.copy()
display(INFERENCE_SETTINGS)


Start with one frame because compilation can dominate the first evaluation. Add frames after measuring cost. Confirmed settings: tf32=True, merge_mole=True, compile=True, activation_checkpointing=False, internal_graph_gen_version=2, external_graph_gen=False.

Explicit units bypass input parsing. No timestep is needed for a snapshot. Every Cartesian component must satisfy the absolute tolerance in eV/angstrom. Match the MD model/task, electronic state, boundaries and ordering.

In [ ]:
report = None
if RUN_VALIDATION:
    if CHARGE is None or SPIN_MULTIPLICITY is None:
        raise ValueError("Set full-frame CHARGE and SPIN_MULTIPLICITY")
    files = resolve_run_files(RUN_DIR, input_file=INPUT_FILE, dump_file=DUMP_FILE, data_file=DATA_FILE, require_input=False)
    print("Selected files:", files)
    report = validate_forces(files["dump"], CHECKPOINT, units=UNITS, datafile=files["data"],
                             indices=FRAME_INDICES, charge=CHARGE, spin=SPIN_MULTIPLICITY,
                             atol=ATOL_EV_PER_ANGSTROM, task=TASK, device=DEVICE,
                             inference_settings=INFERENCE_SETTINGS)
    display(pd.DataFrame(report["frames"]))
    print("All components passed:", report["passed"])
    if REPORT_FILE is not None:
        with Path(REPORT_FILE).open("x",encoding="utf-8") as handle:
            json.dump(report,handle,indent=2)
else:
    print("Set your local model, electronic state, and RUN_VALIDATION=True to evaluate.")
